<a href="https://colab.research.google.com/github/SandhiyaGiri/ComputerVision/blob/main/MNIST_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Write a code using pytorch to Train a two-layer NN that classifies MNIST digits
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms


In [6]:
input_size = 784
hidden_size = 512
num_classes = 10
batch_size = 64
learning_rate = 0.01
num_epochs = 5

In [4]:
# load the data

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.65MB/s]


In [17]:
# Define the classifier

class Classifier(nn.Module):
  def __init__(self, input_size, hidden_size, num_classes) -> None:
    super(Classifier, self).__init__()

    self.fc1 = nn.Linear(input_size, hidden_size)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(hidden_size, num_classes)

  def forward(self, x):
    out = x.view(-1, 28*28)
    out = self.fc1(out)
    out = self.relu(out)
    out = self.fc2(out)
    return out

In [18]:
# train the network
import tqdm

model = Classifier(input_size, hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print( "The device is ", device)
# Training loop
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


The device is  cuda
Epoch [1/5], Loss: 0.1677
Epoch [2/5], Loss: 0.0197
Epoch [3/5], Loss: 0.0055
Epoch [4/5], Loss: 0.1888
Epoch [5/5], Loss: 0.0250


In [19]:
# Testing loop
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy on test set: {100 * correct / total:.2f}%')

Accuracy on test set: 95.90%
